# 02주 · Python 자료형·컬렉션·실행 상태

수치해석및실습 CE273 · 2027학년도 1학기

원본 슬라이드의 개념·예제·차시 순서에 연결한 **신규 병행 실습**. 원본 코드의 단순 복사본과 구분

**목표**: 수치 알고리즘·오차·수렴·안정성 이해와 방법 선택의 근거 탐구

**수업**: 2027-03-09 프로그래밍 (1) · 대면 / 2027-03-11 프로그래밍 (2) · 대면

[Excel 실습](https://raw.githubusercontent.com/steelpl/lecture/main/CE273-numerical-analysis/week02/CE273_W02.xlsx) · [주차 안내](https://github.com/steelpl/lecture/tree/main/CE273-numerical-analysis/week02)

모형·가정 확인 → 손계산·첫 갱신 → 단계별 계산 → 스텝·초기값 변경 → 오차·수렴·안정성 설명

셀 실행: Shift+Enter. 처음에는 위에서 아래로 실행. 공통 준비 셀 실행 후 각 절의 개별 실행도 가능. None 항목과 변경 셀은 직접 작성

핵심 실습: 손계산·원리와 계산식의 연결·조건 변경·해석. AI 활용은 개념 학습 후 선택 활동

## 0. 실행 환경·보조 함수

필요한 버전만 설치. 업데이트한 패키지를 이미 불러온 경우 런타임 재시작 후 처음부터 실행

보조 함수의 세부 문법 암기 불필요. 입력·처리·반환값·오차 기준·실패 조건부터 확인. 아래 실습의 호출 코드와 슬라이드 식 대조

In [ ]:
# 최초 실행 시 필요한 버전만 설치. Colab에서 위에서 아래로 실행
import importlib.metadata as metadata, subprocess, sys
from packaging.version import Version
requirements = {"numpy": "1.26", "pandas": "2.2", "scipy": "1.11", "matplotlib": "3.8", "openpyxl": "3.1"}
for package, minimum in requirements.items():
    try:
        outdated = Version(metadata.version(package)) < Version(minimum)
    except metadata.PackageNotFoundError:
        outdated = True
    if outdated:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", f"{package}>={minimum}"])
print("실행 환경 준비 완료. 이미 불러온 패키지를 업데이트한 경우 런타임 재시작 후 처음부터 실행")

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import optimize, linalg, interpolate, integrate

def vector(values, minimum=1):
    x = np.asarray(values, dtype=float)
    if x.ndim != 1 or len(x) < minimum or not np.isfinite(x).all():
        raise ValueError("유한한 1차원 자료·최소 원소 수 확인 필요")
    return x

def relative_error(truth, estimate):
    # 기준값 0에서 상대오차를 정의하지 않음. 절대오차 별도 확인
    return np.nan if truth == 0 else abs((truth-estimate)/truth)


## S01. 수치·문자열과 공유 상태

**슬라이드 연결**: 2026_S1_NA_Wk2a_Programming1.pptx p4, 5, 16, 17, 20, 24

**자료 구분**: 원본 개념을 적용한 추가 연습

**먼저 예측**: 화면의 소수점 자릿수와 저장 정밀도는 같은가?

단위·스텝·초기값·분모·조건 중 이번 예제의 핵심을 실행 전에 기록

In [ ]:
mass=68.1; coefficient=12.5
print(type(mass),mass/coefficient, f"mass={mass:.1f} kg")
spec={"mass_kg":mass,"drag_kg_s":coefficient}
print(spec)
assert isinstance("68.1",str) and isinstance(68.1,float)
print("출력 표시만 반올림:",f"{1/3:.2f}","계산값:",1/3)

**조건 변경**: mass를 문자열로 바꾼 사본에서 형 변환·단위 누락 확인

원본 셀을 복사한 뒤 한 조건씩 변경. 기준 결과와 변경 결과를 구분

**선택 심화 · AI 검토**: 입력 자료형·단위·출력 형식·셀 실행 순서 명시

**결과 검토**: Colab 셀은 같은 런타임의 변수를 공유. 재시작 후 처음부터 재현

예상·중간 계산·변경 비교·독립 검산·원리 설명 기록

In [ ]:
# 바꾼 조건과 검산 코드를 이 셀에 추가


## S02. count·extend·get과 집합

**슬라이드 연결**: 2026_S1_NA_Wk2b_Programming2.pptx p10, 11, 19, 20, 22, 24

**자료 구분**: 원본 개념을 적용한 추가 연습

**먼저 예측**: len(a)와 a.count(1)의 결과가 같은가?

단위·스텝·초기값·분모·조건 중 이번 예제의 핵심을 실행 전에 기록

In [ ]:
a=[1,2,3,1];print("count(1):",a.count(1),"len:",len(a))
items=[1,2,3];items.extend((4,5));print(items)
d={"m":68.1,"c":12.5};print(list(d),d.get("g",9.81))
try:d["g"]
except KeyError:print("없는 키: KeyError")
assert list(d)==["m","c"] and items==[1,2,3,4,5]
s={3,1,3};print("정렬한 집합 원소:",sorted(s))
assert s=={1,3}

**조건 변경**: extend에 문자열·튜플을 전달한 사본에서 추가 원소 확인

원본 셀을 복사한 뒤 한 조건씩 변경. 기준 결과와 변경 결과를 구분

**선택 심화 · AI 검토**: 컬렉션 종류·순서·수정 여부·키 없음의 처리 방침을 명시

**결과 검토**: dict 삽입순서와 set의 순서 미보장을 구분. count는 대상 인수 필요

예상·중간 계산·변경 비교·독립 검산·원리 설명 기록

In [ ]:
# 바꾼 조건과 검산 코드를 이 셀에 추가


## 주차 확인

자료형·컬렉션의 동작과 런타임 상태 설명

입력 항목:

- `리스트_1의_개수`: 해당 계산값·항 수·오차 기준의 수치
- `extend는_리스트만_허용`: 판단 · Python `True`/`False`
- `dict는_삽입순서를_유지`: 판단 · Python `True`/`False`
- `표시_자릿수와_저장_정밀도가_항상_같음`: 판단 · Python `True`/`False`

공개 계산·개념 확인용 항목. 값만 복사하면 학습 완료로 판단 불가. 가정·적용 범위·검산 근거는 아래 서술과 함께 확인

In [ ]:
answers = {'리스트_1의_개수': None, 'extend는_리스트만_허용': None, 'dict는_삽입순서를_유지': None, '표시_자릿수와_저장_정밀도가_항상_같음': None}


In [ ]:
def check_answers(values):
    expected = {'리스트_1의_개수': 2, 'extend는_리스트만_허용': False, 'dict는_삽입순서를_유지': True, '표시_자릿수와_저장_정밀도가_항상_같음': False}
    pending, wrong = [], []
    for key, target in expected.items():
        got = values.get(key)
        if got is None:
            pending.append(key)
        elif isinstance(target, bool):
            if not isinstance(got, (bool, np.bool_)) or bool(got) != target:
                wrong.append(key)
        elif isinstance(target, str):
            if not isinstance(got, str) or got.strip().lower() != target:
                wrong.append(key)
        elif isinstance(got, (bool, np.bool_)) or not isinstance(got, (int, float, np.number)) or not np.isfinite(got) or not np.isclose(got, target, rtol=1e-6, atol=1e-8):
            wrong.append(key)
    if pending:
        print("미완료:", ", ".join(pending))
    if wrong:
        print("재검토:", ", ".join(wrong))
    if not pending and not wrong:
        print("공개 계산·개념 확인 통과. 해석·가정·AI 검토 기록은 별도 확인 필요")
    return {"pending": pending, "wrong": wrong, "passed": not pending and not wrong}

check_result = check_answers(answers)

## 원리 탐구·결과 해석 기록

1. 문제·자료·단위·모형과 가정
2. 실행 전 예측·작은 손계산과 근거
3. 기준 계산의 중간값·결과와 원리의 연결
4. 한 조건 변경 결과와 차이의 원인
5. 독립 검산·적용 조건·불확실성·한계
6. 자신의 말로 설명할 핵심 원리 한 가지

AI 활용은 선택. 사용 시 요청·의심한 출력·수정·검산 근거를 별도 기록. 글자 수 자동 채점 없음

In [ ]:
review_log = {
    "문제·자료·단위·가정": None,
    "예측·손계산 근거": None,
    "중간 계산·원리 연결": None,
    "조건 변경·차이의 원인": None,
    "검산·해석·적용 한계": None,
    "내 설명": None,
}
# 선택 활동: AI 활용 시 요청·의심한 출력·수정·검산 근거를 별도 기록
